In [ ]:
# All package imports (run this cell first)
import sys
import subprocess
from pathlib import Path

import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import PeftModel, get_peft_model, LoraConfig, TaskType

## Kernel check (use root .venv)

Run this cell first to confirm the notebook is using the project's root `.venv`. The path should contain `My-Crew-Manager\.venv`.

In [ ]:
# Kernel verification: confirm we're using root .venv
_venv_ok = "My-Crew-Manager" in sys.executable and ".venv" in sys.executable
print(f"Python: {sys.executable}")
print(f"Using root .venv: {'✓ Yes' if _venv_ok else '✗ No – select Kernel → Python (My-Crew-Manager .venv)'}")

# My Crew Manager – LLM Training Workflow

End-to-end workflow: generate synthetic dataset → prepare tokenized data → train with LoRA → save adapter → quick inference test.

Run from `AI/` directory or project root. Requires OPENAI_API_KEY or ANTHROPIC_API_KEY for Step 1.

## Setup paths

In [ ]:
# Resolve AI root (notebook may run from AI/ or project root)
_cwd = Path.cwd()
_ai_root = _cwd if (_cwd / "llms").exists() else (_cwd / "AI" if (_cwd / "AI").exists() else _cwd)
if str(_ai_root) not in sys.path:
    sys.path.insert(0, str(_ai_root))

FINE_TUNE_DIR = _ai_root / "llms" / "fine_tune"
DATASET_DIR = FINE_TUNE_DIR / "dataset"
TOKENIZED_DIR = FINE_TUNE_DIR / "tokenized"
OUTPUT_DIR = FINE_TUNE_DIR / "qwen_project_manager_lora"

print(f"AI root: {_ai_root}")
print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")

## Step 1: Generate synthetic dataset

Run build_synthetic to generate proposals and all 6 sections (summary, features, roles, goals, timeline, backlog). Set OPENAI_API_KEY or ANTHROPIC_API_KEY.

In [ ]:
import subprocess

# Generate 15 proposals (use --overwrite to replace existing)
result = subprocess.run(
    [sys.executable, "-m", "llms.fine_tune.build_synthetic", "--count", "15"],
    cwd=str(_ai_root),
    capture_output=False,
)
if result.returncode != 0:
    print("Build synthetic failed. Check API keys and try again.")
else:
    # Show dataset counts
    for name in ["summary", "features", "roles", "goals", "timeline", "backlog"]:
        p = DATASET_DIR / f"{name}.jsonl"
        count = len([ln for ln in p.read_text(encoding="utf-8").split("\n") if ln.strip()]) if p.exists() else 0
        print(f"  {name}.jsonl: {count} examples")

## Step 2: Prepare tokenized dataset

In [ ]:
from llms.fine_tune.prepare_dataset import prepare_tokenized_dataset, load_all_sections

examples = load_all_sections()
print(f"Loaded {len(examples)} examples")

tokenized_path = TOKENIZED_DIR / "tokenized_project_management_qwen"
MAX_LENGTH = 512

if tokenized_path.exists():
    dataset = load_from_disk(str(tokenized_path))
    print(f"Loaded tokenized dataset from {tokenized_path}")
else:
    dataset = prepare_tokenized_dataset(
        model_name="qwen",
        max_length=MAX_LENGTH,
        output_dir=str(tokenized_path),
    )
print(f"Dataset size: {len(dataset)}")

## Step 3: Load model & apply LoRA

In [ ]:
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
BATCH_SIZE = 2
EPOCHS = 3

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
if torch.cuda.is_available():
    model = model.to("cuda")

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 4: Train

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()

## Step 5: Save adapter

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Saved adapter and tokenizer to {OUTPUT_DIR}")
print("Set PEFT_ADAPTER_PATH in AI/.env to use this adapter:")
print("  PEFT_ADAPTER_PATH=llms/fine_tune/qwen_project_manager_lora")

## Step 6: Quick inference test

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model_infer = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
if torch.cuda.is_available():
    model_infer = model_infer.to("cuda")

test_prompt = "You are a senior AI assistant. Read the proposal and generate a summary.\n\n<<<PROPOSAL>>>\nBuild a task management web app for small teams.\n<<<END PROPOSAL>>>\n\nsummary:"
inputs = tokenizer(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}
outputs = model_infer.generate(**inputs, max_new_tokens=64, do_sample=True, temperature=0.4)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Generated response:")
print(response.strip())